In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("miadul/brain-tumor-dataset")

import os

# Listar archivos en la ruta descargada
print("Archivos descargados:")
print(os.listdir(path))

Archivos descargados:
['brain_tumor_dataset.csv']


In [3]:
import pandas as pd

# Cargar el archivo en un DataFrame 
df = pd.read_csv(os.path.join(path, "brain_tumor_dataset.csv"))

df = df.set_index("Patient_ID")

# EDA

In [4]:
df.head()

,Age,Gender,Tumor_Type,Tumor_Size,Location,Histology,Stage,Symptom_1,Symptom_2,Symptom_3,Radiation_Treatment,Surgery_Performed,Chemotherapy,Survival_Rate,Tumor_Growth_Rate,Family_History,MRI_Result,Follow_Up_Required
Patient_ID,,,,,,,,,,,,,,,,,,
1,73,Male,Malignant,5.375612,Temporal,Astrocytoma,III,Vision Issues,Seizures,Seizures,No,No,No,51.312579,0.111876,No,Positive,Yes
2,26,Male,Benign,4.847098,Parietal,Glioblastoma,II,Headache,Headache,Nausea,Yes,Yes,Yes,46.373273,2.165736,Yes,Positive,Yes
3,31,Male,Benign,5.588391,Parietal,Meningioma,I,Vision Issues,Headache,Seizures,No,No,No,47.072221,1.884228,No,Negative,No
4,29,Male,Malignant,1.436600,Temporal,Medulloblastoma,IV,Vision Issues,Seizures,Headache,Yes,No,Yes,51.853634,1.283342,Yes,Negative,No
5,54,Female,Benign,2.417506,Parietal,Glioblastoma,I,Headache,Headache,Seizures,No,No,Yes,54.708987,2.069477,No,Positive,Yes


In [5]:
df.columns

Index(['Age', 'Gender', 'Tumor_Type', 'Tumor_Size', 'Location', 'Histology',
       'Stage', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Radiation_Treatment',
       'Surgery_Performed', 'Chemotherapy', 'Survival_Rate',
       'Tumor_Growth_Rate', 'Family_History', 'MRI_Result',
       'Follow_Up_Required'],
      dtype='object')

In [6]:
list(set(df["Location"]))

['Temporal', 'Occipital', 'Frontal', 'Parietal']

In [7]:
list(set(df["Histology"]))

['Astrocytoma', 'Medulloblastoma', 'Glioblastoma', 'Meningioma']

In [8]:
list(set(df["Stage"]))

['II', 'III', 'IV', 'I']

In [9]:
list(set(df["Symptom_1"]))

['Seizures', 'Vision Issues', 'Headache', 'Nausea']

In [10]:
list(set(df["Symptom_2"]))

['Seizures', 'Vision Issues', 'Headache', 'Nausea']

In [11]:
list(set(df["Symptom_3"]))

['Seizures', 'Vision Issues', 'Headache', 'Nausea']

# FE

In [12]:
num_columns = [
    'Age',
    'Tumor_Size',
    'Survival_Rate',
    'Tumor_Growth_Rate'
]

cat_columns = [
    'Gender',
    #'Location',
    'Histology',
    'Stage',
    #'Symptom_1',
    #'Symptom_2',
    #'Symptom_3',
    #'Radiation_Treatment',
    'Surgery_Performed',
    'Chemotherapy',
    #'Family_History',
    'MRI_Result',
    'Follow_Up_Required'
]

target = 'Tumor_Type'

X = df[num_columns + cat_columns]
y = df[target]

In [13]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder # Escalar / Dummies
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, num_columns),
        ('cat', categorical_transformer, cat_columns)
    ]
)

# Modelo

In [14]:
from sklearn.svm import SVC
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.metrics import roc_auc_score, make_scorer
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [15]:
def objective(trial):
    # Hiperparámetros a optimizar
    C = trial.suggest_loguniform('C', 1e-4, 1e2)
    
    # Crear modelo
    model = SVC(
        kernel='linear',
        C=C,
        probability=True,   
        random_state=42
    )
    
    # Crear pipeline
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    # Definir K-Folds
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    
    # Calcular AUC promedio
    auc_scores = cross_val_score(pipeline, X.head(200), y.head(200), cv=cv, scoring='roc_auc', n_jobs=-1)
    
    return auc_scores.mean()

In [16]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=5, n_jobs=1)

In [17]:
print("Mejor hiperparámetro encontrado:")
print(study.best_params)

print(f"Mejor AUC promedio: {study.best_value:.4f}")

Mejor hiperparámetro encontrado:
{'C': 0.00015090994801366655}
Mejor AUC promedio: 0.5736
